# Project Track 1 - Reynolds-Number Generalization

**Choose one variant:**

- **1A Interpolation:** quantify performance at unseen Reynolds numbers inside the training range.
- **1B Extrapolation:** train only on lower-Re cases and determine where the surrogate fails at higher Re.

The notebook supplies the solver output, interpolation baseline, coordinate DNN, metrics, and plotting functions. Your work is to freeze a split, select an architecture using validation only, perform a controlled comparison, and defend a conclusion.

## Required files
`P1_Re_Generalization.ipynb`, `w4utils.py`, `w5_common.py`, `cavity_data.npz`

<!-- MIE690A enriched learner edition v2 -->

## How to learn from this notebook

This is a guided computational laboratory, not a script to execute without reading. For every numbered stage:

1. read the physical question and write a prediction;
2. inspect the inputs, outputs, units, and split before running code;
3. run the cell and check assertions/warnings;
4. compare with the stated baseline or physical diagnostic; and
5. write one or two sentences explaining what the result does **and does not** establish.

Use **Restart and Run All** before treating any output as final. Hidden state from out-of-order execution is a reproducibility failure.

### Evidence contract

Keep four kinds of evidence separate:

- **numerical evidence:** residuals, accepted cases, data hashes, grid/time/particle budgets;
- **statistical evidence:** losses, relative errors, variability across seeds/cases;
- **physical evidence:** centerlines, walls, divergence, vortex structure, positivity, moments;
- **computational evidence:** runtime, memory, saved configuration, and machine-readable metrics.

A claim is only as strong as the weakest relevant layer.


## Learning objectives and prerequisites

By the end you should be able to distinguish interpolation from extrapolation geometrically, create a case-wise split, select an architecture without test leakage, retrain fairly on all permitted development cases, and localize the first tested Reynolds interval where a failure threshold is crossed.

Prerequisites: Week-4 data audit, feature/target scaling, relative L2 error, centerline interpretation, and the difference between model selection and final evaluation.


In [ ]:
# Repository/Colab bootstrap. Run this before the import cell below.
from pathlib import Path
import sys

def _find_course_root(start=Path.cwd()):
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "common" / "w5_common.py").exists():
            return base
    # Colab flat-upload fallback: helper files and data beside the notebook.
    if (start / "w5_common.py").exists():
        return start
    raise FileNotFoundError(
        "Course root not found. Clone the repository, or upload w4utils.py, "
        "w5_common.py, the track-specific helpers, and cavity_data.npz as listed above."
    )

COURSE_ROOT = _find_course_root()
COMMON_DIR = COURSE_ROOT / "common" if (COURSE_ROOT / "common").exists() else COURSE_ROOT
DATASET_PATH = COURSE_ROOT / "data" / "cavity_data.npz"
if not DATASET_PATH.exists():
    DATASET_PATH = COURSE_ROOT / "cavity_data.npz"
sys.path.insert(0, str(COMMON_DIR))
print("Course root:", COURSE_ROOT)
print("Common helpers:", COMMON_DIR)
print("Dataset:", DATASET_PATH)


In [ ]:
import time, importlib, ast
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import w4utils, w5_common
importlib.reload(w4utils); importlib.reload(w5_common)
w5_common.set_global_seed(690)
assert w4utils.W4_UTILS_VERSION == "6.3", "Upload the revised w4utils.py (v6.3)."
assert w5_common.W5_COMMON_VERSION == "1.2", "Upload the revised w5_common.py (v1.2)."
data=w5_common.require_week4_files(str(DATASET_PATH))


## Generalization is defined by the physical support

For Variant 1A, every blind Reynolds number lies within the convex interval spanned by development Reynolds numbers. For Variant 1B, blind cases lie above the largest development Reynolds number. The neural network sees coordinates from every part of the cavity in both variants; what changes is the operating condition.

This is why a random point-wise split is misleading: neighboring points from the same Reynolds realization leak its structure into both training and test sets.

**Prediction prompt:** Draw the development Reynolds numbers on a line. Mark every blind case. State whether each is interpolation or extrapolation before running any model.


## 1. Freeze the scientific split before training

The two variants intentionally answer different questions. Do not move a test case into training after viewing its result.

In [ ]:
VARIANT="1A"  # EDIT to "1B" only if Track 1B was approved.

if VARIANT=="1A":
    TRAIN_RE=[100,150,200,250,300,400]
    VAL_RE=225
    TEST_RE=[175,275,350,375]
    CLAIM="interpolation inside the covered Reynolds-number range"
elif VARIANT=="1B":
    TRAIN_RE=[100,150,200,225,250]
    VAL_RE=300
    TEST_RE=[350,375,400]
    CLAIM="higher-Re extrapolation beyond the training range"
else:
    raise ValueError("VARIANT must be 1A or 1B")

print(CLAIM); print("train",TRAIN_RE,"validation",VAL_RE,"blind test",TEST_RE)


## What the architecture comparison can establish

The candidate sweep tests a restricted question: with fixed data, optimizer, stopping logic, and seed, which width/depth gives the best validation evidence? It does not establish a universal best architecture.

Read the selection code in five steps: build case-wise samples; fit scalers on training data; train each candidate; evaluate one complete validation case; select and record the epoch budget. If a scaler is fit using blind coordinates/targets, the experiment is contaminated even when the network never directly trains on blind labels.


## 2. Architecture selection is a validation experiment

All candidates use the same data, optimizer, and stopping rule. Select by Re=`VAL_RE`; never select by blind-test error.

In [ ]:
CANDIDATES=[(32,32),(64,64,64),(96,96)]
selection=[]; bundles={}
for hidden in CANDIDATES:
    t0=time.time()
    b=w5_common.train_pointwise_model(data,TRAIN_RE,VAL_RE,hidden=hidden,
        stride=2,seed=690,epochs=850,patience=60)
    pred=w5_common.predict_case(b,VAL_RE,data["x"],data["y"])
    rep=w5_common.evaluate_prediction(data,VAL_RE,pred)
    selection.append({"hidden":str(hidden),"best_epoch":b["best_epoch"],
                      "val_relative_L2_uv":rep["relative_L2_uv"],
                      "val_relative_L2_p":rep["relative_L2_p"],
                      "seconds":time.time()-t0})
    bundles[hidden]=b
selection=pd.DataFrame(selection).sort_values("val_relative_L2_uv")
display(selection)
BEST=ast.literal_eval(selection.iloc[0]["hidden"])
FINAL_EPOCHS=int(selection.iloc[0]["best_epoch"])
print("Frozen architecture:",BEST)
print("Frozen full-development epoch budget:",FINAL_EPOCHS)


## 3. Retrain once on **all** permitted development cases

After architecture and epoch budget are frozen using `VAL_RE`, add the validation case to the development set and retrain once with a fixed number of epochs. No development Reynolds number is held out during this final fit. This gives the DNN and the field-interpolation baseline exactly the same Reynolds-number information. Blind-test cases remain unopened.

In [ ]:
DEV_RE=sorted(TRAIN_RE+[VAL_RE])
final_bundle=w5_common.train_pointwise_fixed_epochs(
    data,DEV_RE,hidden=BEST,stride=2,seed=690,
    epochs=FINAL_EPOCHS,learning_rate=1e-3)
print("Final DNN trained on all development cases:",DEV_RE)


## Metric glossary for the blind comparison

- `relative_L2_uv`: global velocity-vector error; useful but dominated by large smooth regions.
- `relative_L2_p`: pressure-field error after a consistent gauge; does not replace pressure-gradient inspection.
- `wall_rms`: boundary fidelity with the two lid corners excluded because the ideal boundary condition is discontinuous there.
- `divergence_rms`: discrete incompressibility diagnostic; a coordinate DNN is not divergence-free by construction.
- centerlines: local profiles that reveal misplaced extrema and curvature.
- vortex evidence: structure and location that can be hidden by a small global norm.

Predeclare a failure threshold. Crossing it localizes the failure to an interval between tested cases; it does not reveal an exact critical Reynolds number.


## 4. Open the blind cases and compare fairly against field interpolation

Both methods below use the same `DEV_RE` cases. The required comparison is **not** just a global RMSE table. Include centerline, wall, divergence, and pressure-interior evidence.

**Important expectation:** for this smooth, fixed-grid cavity family, direct field interpolation is a very strong baseline and may outperform the DNN by a wide margin. A well-supported negative result is scientifically valid; do not tune the blind cases simply to make the neural model win.

In [ ]:
rows=[]; stored={}
for r in TEST_RE:
    interp=w5_common.interpolate_case(data,r,DEV_RE)
    dnn=w5_common.predict_case(final_bundle,r,data["x"],data["y"])
    stored[r]={"interpolation":interp,"coordinate DNN":dnn}
    for method,pred in stored[r].items():
        rows.append({"variant":VARIANT,"method":method,"Re":r,
                     **w5_common.evaluate_prediction(data,r,pred)})
results=w5_common.results_frame(rows)
results.to_csv("P1_results.csv",index=False)
display(results)


In [ ]:
# Select the most informative blind case, not merely the prettiest result.
CASE_TO_PLOT=TEST_RE[-1] if VARIANT=="1B" else 275
fig=w5_common.plot_case_evidence(data,CASE_TO_PLOT,stored[CASE_TO_PLOT],
                                 f"Track {VARIANT}: blind evidence")
plt.show()


## Required student decisions and report evidence

1. Explain why your split measures interpolation or extrapolation.
2. Show the validation-only architecture table.
3. Report both interpolation and DNN metrics for every blind case.
4. Include two velocity centerlines and at least one pressure profile.
5. Use wall error and divergence as physical checks.
6. Identify the first Reynolds number at which the model becomes unreliable under a criterion you state in advance.
7. Explain whether the neural model justifies its complexity over interpolation.

**Do not:** change the split after test results, tune on Re=375/400, or claim generalization from one contour plot.

**Fairness check:** confirm in the report that both the DNN and interpolation baseline used exactly the same `DEV_RE` list.

## Optional stretch extensions (advanced / prize-track only)

Complete the required project first. With instructor approval, choose at most one:

1. Train a small ensemble and test whether ensemble spread rises near the observed Reynolds-number failure boundary.
2. Compare a validation-derived rejection rule against actual blind error and discuss false confidence.
3. Compare interpolation, the coordinate DNN, and one reduced-order coefficient model under the same development cases and storage/runtime accounting.


## Concept check and further reading

1. Why can direct field interpolation use fewer modeling assumptions than a coordinate DNN?
2. Why is Re = 350 only the *first tested failure* in a grid containing 300 and 350?
3. Which metric is most sensitive to a moving-lid corner error?
4. What evidence would justify the added neural complexity?
5. If the DNN wins globally but loses on divergence, how should the conclusion be written?

Read Brunton, Noack & Koumoutsakos (2020) for ML task structure in fluids and Wilson et al. (2014) for controlled computational comparisons.


## Reproducibility record

Before closing the notebook, record:

- Python and package versions;
- dataset hash and helper versions;
- every physical case in development, validation, and blind sets;
- every seed and candidate value tried;
- the selection rule and when it was frozen;
- output filenames and units; and
- any cell that was skipped, changed, or run with a reduced budget.

Restart the kernel and run all cells in order. If the result changes materially, report the variability instead of selecting the preferred run.

## Troubleshooting without corrupting the experiment

| Symptom | Safe action | Unsafe action |
| --- | --- | --- |
| Missing helper/data file | Re-run the bootstrap and verify paths/hash | Download an unlabeled older copy |
| Training is slow | Use the documented smoke configuration, then label it “smoke” | Quietly reduce epochs/data in the final claim |
| Validation is poor | Inspect scaling, split, and baseline; revise on development data | Open the blind case to choose settings |
| Blind result fails | Report/localize failure and propose a new future experiment | Tune on the blind case while keeping its label “blind” |
| Stochastic result changes | Run multiple declared seeds and report mean/spread | Keep rerunning until one result looks good |
| A neural model loses to interpolation | Verify fairness, then recommend the simpler method | Hide the baseline |

## Final report outline

1. **Question and hypothesis** — one falsifiable sentence.
2. **Data and split** — physical cases, numerical source, and blind unit.
3. **Baseline** — simplest credible comparator using the same allowed information.
4. **Modification** — the one controlled change.
5. **Selection** — validation-only candidates and frozen rule.
6. **Blind numerical result** — aggregate errors and variability.
7. **Physical result** — at least two diagnostics tied to the flow.
8. **Failure or limitation** — where confidence ends.
9. **Cost and reproducibility** — runtime, environment, seeds, saved files.
10. **Conclusion** — helped, hurt, or revealed a tradeoff; no forced positive AI claim.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Figure 9 and neural summary: controlled Reynolds-number generalization.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../ARTICLE_FIGURE_MAP.md).
